In [ ]:
!pip install bitsandbytes

In [ ]:
!pip install peft

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 411.1/411.1 kB 8.1 MB/s eta 0:00:00


In [ ]:
!pip install datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 15.0 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.5.1
    Uninstalling fsspec-2025.5.1:
      Successfully uninstalled fsspec-2025.5.1


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
path = r"./drive/MyDrive/edge_med_project/"

In [3]:
import torch
import torch.nn.functional as F
from transformers import (
    GPT2Config, GPT2LMHeadModel, GPT2Tokenizer,
    Trainer, TrainingArguments, DataCollatorForLanguageModeling, AutoModelForCausalLM, AutoTokenizer
)
from peft import PeftModel, PeftConfig
from datasets import Dataset
import pandas as pd
from typing import Dict, Any, Optional, Union

In [4]:
# =============================================================================
# 1. STUDENT MODEL CONFIGURATION (196M parameters)
# =============================================================================

def create_student_model(model_name_or_path=None):
    """Create the 196M parameter student model"""

    # Your custom configuration: 1024 embedding, 11 layers, 16 heads
    student_config = GPT2Config(
        vocab_size=50257,        # Standard GPT-2 vocab
        n_positions=512,         # Context length for your explanations
        n_embd=1024,            # Embedding dimension (width)
        n_layer=11,             # Number of transformer layers
        n_head=16,              # Attention heads (robust attention)
        n_inner=4096,           # Feed-forward size (4x embedding)
        activation_function="gelu_new",
        resid_pdrop=0.1,
        embd_pdrop=0.1,
        attn_pdrop=0.1,
        layer_norm_epsilon=1e-5,
        initializer_range=0.02,
        summary_type="cls_index",
        summary_use_proj=True,
        summary_activation=None,
        summary_proj_to_labels=True,
        summary_first_dropout=0.1,
        scale_attn_weights=True,
        use_cache=True,
        bos_token_id=50256,
        eos_token_id=50256,
    )

    # Create the model
    if model_name_or_path:
        student_model = GPT2LMHeadModel.from_pretrained(model_name_or_path, config=student_config)
    else:
        student_model = GPT2LMHeadModel(student_config)

    print(f"Student model configuration:")
    print(f"  Total Parameters: {student_model.num_parameters()/1e6:.2f}M")
    print(f"  Embedding dim: {student_config.n_embd}")
    print(f"  Layers: {student_config.n_layer}")
    print(f"  Attention heads: {student_config.n_head}")
    print(f"  Context length: {student_config.n_positions}")

    return student_model, student_config

In [5]:
# =============================================================================
# 2. TEACHER MODEL LOADING
# =============================================================================

def load_teacher_model(base_model_id, adapter_path):
    """Load your fine-tuned Phi-4 teacher model"""

    print("Loading teacher model (Phi-4 + LoRA)...")

    # Load base model with 4-bit quantization
    from transformers import AutoModelForCausalLM, AutoTokenizer

    base_model = AutoModelForCausalLM.from_pretrained(
        base_model_id,
        torch_dtype=torch.bfloat16,
        device_map='auto',
        load_in_4bit=True,
        trust_remote_code=True
    )

    # Apply LoRA adapters
    teacher_model = PeftModel.from_pretrained(base_model, adapter_path)
    teacher_tokenizer = AutoTokenizer.from_pretrained(base_model_id)

    if teacher_tokenizer.pad_token is None:
        teacher_tokenizer.pad_token = teacher_tokenizer.eos_token

    print("✅ Teacher model loaded successfully")
    return teacher_model, teacher_tokenizer


In [3]:
def load_finetuned_model(adapter_path, base_model):

  # Load the adapter config to get base model info
  peft_config = PeftConfig.from_pretrained(adapter_path)

  # Load base model
  base_model = AutoModelForCausalLM.from_pretrained(
      peft_config.base_model_name_or_path,
      torch_dtype=torch.bfloat16,
      device_map='auto',
      trust_remote_code=True
  )

  # Load and apply adapters
  model = PeftModel.from_pretrained(base_model, adapter_path)

  # Load tokenizer
  tokenizer = AutoTokenizer.from_pretrained(peft_config.base_model_name_or_path)
  if tokenizer.pad_token is None:
      tokenizer.pad_token = tokenizer.eos_token

  return model, tokenizer

In [21]:
# =============================================================================
# 3. DATA PREPARATION
# =============================================================================

def prepare_distillation_dataset(df, tokenizer, max_length=512):
    """
    Prepare dataset for distillation with proper padding handling
    Expected df columns: 'DHS Code', 'Description', 'explanation'
    """

    formatted_data = []

    for _, row in df.iterrows():
        code = row['DHS Code']
        description = row['Description']
        explanation = row['explanation']

        # Create input prompt (what both teacher and student see)
        input_prompt = f"Explain the medical code {code}: {description}\nExplanation:"

        # Create full sequence (input + target explanation)
        full_sequence = input_prompt + " " + explanation

        formatted_data.append({
            "input_prompt": input_prompt,
            "full_sequence": full_sequence,
            "explanation": explanation
        })

    # Convert to HuggingFace dataset
    dataset = Dataset.from_list(formatted_data)

    # Tokenize with consistent padding
    def tokenize_function(examples):
        # Tokenize input prompts (for teacher generation)
        input_encodings = tokenizer(
            examples["input_prompt"],
            truncation=True,
            padding="max_length",  # Pad to max_length for consistency
            max_length=max_length//2,  # Leave room for generation
            padding_side='left',
            return_tensors=None
        )

        # Tokenize full sequences (for student training targets)
        full_encodings = tokenizer(
            examples["full_sequence"],
            truncation=True,
            padding="max_length",  # Pad to max_length for consistency
            max_length=max_length,
            padding_side='left',
            return_tensors=None
        )

        return {
            "input_ids": input_encodings["input_ids"],
            "attention_mask": input_encodings["attention_mask"],
            "target_ids": full_encodings["input_ids"],
            "target_attention_mask": full_encodings["attention_mask"],
            "input_length": [len([x for x in ids if x != tokenizer.pad_token_id]) for ids in input_encodings["input_ids"]]
        }

    dataset = dataset.map(tokenize_function, batched=True, remove_columns=dataset.column_names)

    print(f"✅ Dataset prepared: {len(dataset)} examples")
    print(f"   Sequence length: {max_length} tokens (padded)")
    return dataset

In [8]:
class DistillationDataCollator:
    """Custom data collator for distillation with proper padding"""

    def __init__(self, tokenizer, pad_to_multiple_of=8):
        self.tokenizer = tokenizer
        self.pad_to_multiple_of = pad_to_multiple_of

    def __call__(self, features):
        # Convert lists to tensors and ensure consistent shapes
        batch = {}

        # Get max lengths in this batch (even though we pre-padded)
        max_input_length = max(len(f["input_ids"]) for f in features)
        max_target_length = max(len(f["target_ids"]) for f in features)

        # Pad to multiple of 8 for efficiency (optional)
        if self.pad_to_multiple_of:
            max_input_length = ((max_input_length + self.pad_to_multiple_of - 1)
                              // self.pad_to_multiple_of * self.pad_to_multiple_of)
            max_target_length = ((max_target_length + self.pad_to_multiple_of - 1)
                               // self.pad_to_multiple_of * self.pad_to_multiple_of)

        # Prepare batch tensors
        batch_size = len(features)

        batch["input_ids"] = torch.full((batch_size, max_input_length),
                                      self.tokenizer.pad_token_id, dtype=torch.long)
        batch["attention_mask"] = torch.zeros((batch_size, max_input_length), dtype=torch.long)
        batch["target_ids"] = torch.full((batch_size, max_target_length),
                                       self.tokenizer.pad_token_id, dtype=torch.long)
        batch["target_attention_mask"] = torch.zeros((batch_size, max_target_length), dtype=torch.long)
        batch["input_length"] = torch.tensor([f["input_length"] for f in features], dtype=torch.long)

        # Fill in the actual data
        for i, feature in enumerate(features):
            input_len = len(feature["input_ids"])
            target_len = len(feature["target_ids"])

            batch["input_ids"][i, :input_len] = torch.tensor(feature["input_ids"])
            batch["attention_mask"][i, :input_len] = torch.tensor(feature["attention_mask"])
            batch["target_ids"][i, :target_len] = torch.tensor(feature["target_ids"])
            batch["target_attention_mask"][i, :target_len] = torch.tensor(feature["target_attention_mask"])

        return batch


In [9]:
# =============================================================================
# 5. CUSTOM DISTILLATION TRAINER
# =============================================================================

class DistillationTrainer(Trainer):
    """Custom trainer for knowledge distillation"""

    def __init__(
        self,
        teacher_model,
        teacher_tokenizer,
        alpha=0.7,
        temperature=1.0,
        **kwargs
    ):
        super().__init__(**kwargs)
        self.teacher_model = teacher_model
        self.teacher_tokenizer = teacher_tokenizer
        self.alpha = alpha  # Weight for KL divergence loss
        self.temperature = temperature  # Temperature for soft targets

        # Set teacher to eval mode
        self.teacher_model.eval()

        print(f"✅ Distillation trainer initialized:")
        print(f"   Alpha (KL weight): {self.alpha}")
        print(f"   Temperature: {self.temperature}")

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        """
        Custom loss computation with KL divergence + cross entropy
        """

        # Get student outputs on full sequence
        student_outputs = model(
            input_ids=inputs["target_ids"],
            attention_mask=inputs["target_attention_mask"],
            labels=inputs["target_ids"]  # For CE loss computation
        )

        student_logits = student_outputs.logits
        student_loss = student_outputs.loss  # Standard CE loss

        # Generate teacher outputs on-the-fly
        with torch.no_grad():
            # Teacher generates from input prompt
            teacher_generation = self.teacher_model.generate(
                input_ids=inputs["input_ids"],
                attention_mask=inputs["attention_mask"],
                max_new_tokens=100,  # Reasonable explanation length
                temperature=self.temperature,
                do_sample=True,
                pad_token_id=self.teacher_tokenizer.eos_token_id,
                eos_token_id=self.teacher_tokenizer.eos_token_id,
                return_dict_in_generate=True,
                output_scores=True
            )

            # Get teacher logits for the generated sequence
            teacher_outputs = self.teacher_model(
                input_ids=teacher_generation.sequences,
                attention_mask=torch.ones_like(teacher_generation.sequences)
            )
            teacher_logits = teacher_outputs.logits

        # Compute KL divergence loss on explanation part only
        kl_loss = self.compute_kl_loss(student_logits, teacher_logits, inputs)

        # Combined loss: alpha * KL + (1-alpha) * CE
        total_loss = self.alpha * kl_loss + (1 - self.alpha) * student_loss

        if return_outputs:
            return total_loss, student_outputs
        return total_loss

    def compute_kl_loss(self, student_logits, teacher_logits, inputs):
        """
        Compute KL divergence loss, masking input tokens and padding tokens
        """

        # Get minimum sequence length
        min_length = min(student_logits.size(1), teacher_logits.size(1))

        # Truncate to same length
        student_logits = student_logits[:, :min_length, :]
        teacher_logits = teacher_logits[:, :min_length, :]

        # Create mask for explanation tokens only (exclude input prompt AND padding)
        batch_size = student_logits.size(0)
        loss_mask = torch.zeros((batch_size, min_length), device=student_logits.device)

        for i in range(batch_size):
            input_length = inputs["input_length"][i].item()

            # Find actual sequence end (before padding)
            target_attention = inputs["target_attention_mask"][i][:min_length]
            actual_length = target_attention.sum().item()

            if actual_length > input_length:
                # Only compute loss on explanation part (after input, before padding)
                loss_mask[i, input_length:actual_length] = 1.0

        # Apply temperature scaling
        student_probs = F.log_softmax(student_logits / self.temperature, dim=-1)
        teacher_probs = F.softmax(teacher_logits / self.temperature, dim=-1)

        # Compute KL divergence
        kl_div = F.kl_div(
            student_probs,
            teacher_probs,
            reduction='none'
        ).sum(dim=-1)  # [batch_size, seq_len]

        # Apply mask and average
        masked_kl = kl_div * loss_mask

        # Average over valid tokens
        num_valid_tokens = loss_mask.sum()
        if num_valid_tokens > 0:
            kl_loss = masked_kl.sum() / num_valid_tokens
        else:
            kl_loss = torch.tensor(0.0, device=student_logits.device)

        # Apply temperature scaling factor
        kl_loss = kl_loss * (self.temperature ** 2)

        return kl_loss


In [10]:
def fix_vocabulary_sizes(student_model, teacher_model, tokenizer):
    """Fix the vocabulary size mismatch"""

    print("🔧 Fixing vocabulary size mismatch...")

    # Use teacher's actual vocabulary size as the target
    target_vocab_size = teacher_model.get_input_embeddings().num_embeddings  # 200064

    print(f"Target vocabulary size: {target_vocab_size}")
    print(f"Current student size: {student_model.get_input_embeddings().num_embeddings}")

    # Resize student model to match teacher
    student_model.resize_token_embeddings(target_vocab_size)

    print(f"✅ Student model resized to: {student_model.get_input_embeddings().num_embeddings}")

    # Verify they match now
    teacher_size = teacher_model.get_input_embeddings().num_embeddings
    student_size = student_model.get_input_embeddings().num_embeddings

    print(f"Teacher size: {teacher_size}")
    print(f"Student size: {student_size}")
    print(f"Match: {teacher_size == student_size}")

    return student_model

# Apply the fix before training


In [11]:
def setup_distillation_training(
    student_model,
    teacher_model,
    teacher_tokenizer,
    train_dataset,
    output_dir="./distilled-medical-explainer",
    **training_kwargs
):
    """Setup training arguments and data collator"""

    # Training arguments optimized for distillation
    training_args = TrainingArguments(
        output_dir=output_dir,
        overwrite_output_dir=True,

        # Batch size and gradient accumulation
        per_device_train_batch_size=2,  # Start small, increase if memory allows
        gradient_accumulation_steps=4,   # Effective batch size = 8

        # Learning and optimization
        learning_rate=5e-5,  # Lower LR for distillation
        weight_decay=0.01,
        max_grad_norm=1.0,

        # Training schedule
        num_train_epochs=3,
        warmup_steps=100,
        lr_scheduler_type="cosine",

        # Memory optimization
        gradient_checkpointing=True,
        dataloader_pin_memory=False,

        # Logging and saving
        logging_steps=10,
        save_steps=500,
        save_total_limit=2,

        # Mixed precision
        bf16=True,  # Use bfloat16 for stability

        # Other
        remove_unused_columns=False,  # Keep our custom columns
        report_to=None,  # Disable wandb for now

        **training_kwargs
    )

    # Use our custom data collator instead of standard one
    data_collator = DistillationDataCollator(
        tokenizer=teacher_tokenizer,
        pad_to_multiple_of=8
    )

    print("✅ Training setup completed with custom data collator")
    return training_args, data_collator

In [12]:
student_model, student_config = None, None

In [13]:
teacher_model, teacher_tokenizer = None, None

In [14]:
train_dataset = None
training_args = None
data_collator = None
trainer = None

In [15]:
student_model

In [16]:
def train_distillation_model(
    df,  # Your dataframe with medical codes, descriptions, explanations
    teacher_model_id="microsoft/Phi-4-mini-instruct",
    teacher_adapter_path="./phi4-medical-lora",
    output_dir="./distilled-medical-explainer"
):
    """
    Main function to train your distilled medical explanation model
    """
    global student_model, student_config, teacher_model, teacher_tokenizer, train_dataset, training_args, data_collator, trainer

    print("🚀 Starting Medical Explanation Model Distillation")
    print("=" * 60)

    # 1. Create student model
    print("1. Creating student model...")
    student_model, student_config = create_student_model()

    print("2. Loading teacher model...")

    # 2. Load teacher model
    if teacher_model and teacher_tokenizer:
      print("   Teacher model already loaded")
    else:
      print("   Loading teacher model...")
      teacher_model, teacher_tokenizer = load_finetuned_model(
          teacher_adapter_path,
          None
      )
    # 3. Setup student tokenizer (same as teacher for compatibility)
    print("3. Setting up tokenizers...")
    student_tokenizer = teacher_tokenizer  # Use same tokenizer

    # Ensure pad_token is set properly
    if student_tokenizer.pad_token is None:
        student_tokenizer.pad_token = student_tokenizer.eos_token
        student_tokenizer.pad_token_id = student_tokenizer.eos_token_id
        print(f"   Set pad_token_id to: {student_tokenizer.pad_token_id}")

    # Resize student embeddings if vocab sizes differ

    if student_model.config.vocab_size != len(student_tokenizer):
        student_model.resize_token_embeddings(len(student_tokenizer))
        print(f"   Resized student embeddings to: {len(student_tokenizer)}")
    student_model = fix_vocabulary_sizes(student_model, teacher_model, teacher_tokenizer)

    # 4. Prepare dataset
    print("4. Preparing dataset...")
    train_dataset = prepare_distillation_dataset(df, student_tokenizer)

    # 5. Setup training
    print("5. Setting up training...")
    training_args, data_collator = setup_distillation_training(
        student_model, teacher_model, teacher_tokenizer, train_dataset, output_dir
    )

    # 6. Create distillation trainer
    print("6. Creating distillation trainer...")
    trainer = DistillationTrainer(
        model=student_model,
        teacher_model=teacher_model,
        teacher_tokenizer=teacher_tokenizer,
        args=training_args,
        train_dataset=train_dataset,
        data_collator=data_collator,
        tokenizer=student_tokenizer,
        alpha=0.7,        # Your choice: 70% KL divergence
        temperature=1.0   # Your choice: temperature 1.0
    )

    # 7. Start training
    print("7. Starting distillation training...")
    print(f"   Dataset size: {len(train_dataset)}")
    print(f"   Estimated training time: ~2-4 hours")

    trainer.train()

    # 8. Save final model
    print("8. Saving distilled model...")
    trainer.save_model()
    student_tokenizer.save_pretrained(output_dir)

    print("✅ Distillation training completed!")
    print(f"   Model saved to: {output_dir}")

    return trainer, student_model, student_tokenizer

In [17]:
# =============================================================================
# 7. USAGE EXAMPLE
# =============================================================================

if __name__ == "__main__":

    # Example usage
    print("Example usage:")
    print("""
    # Load your dataset
    df = pd.read_csv('your_medical_data.csv')
    # Expected columns: 'DHS Code', 'Description', 'explanation'

    # Train distilled model
    trainer, student_model, tokenizer = train_distillation_model(
        df=df,
        teacher_model_id="microsoft/Phi-4-mini-instruct",
        teacher_adapter_path="./phi4-medical-lora",
        output_dir="./distilled-medical-explainer"
    )

    # Test the distilled model
    prompt = "Explain the medical code A00.0: Cholera due to Vibrio cholerae\\nExplanation:"
    inputs = tokenizer(prompt, return_tensors="pt")

    with torch.no_grad():
        outputs = student_model.generate(
            inputs.input_ids,
            max_new_tokens=80,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    print(f"Student model explanation: {response}")
    """)
    df = pd.read_csv(path + r"progress.csv")


Example usage:

    # Load your dataset
    df = pd.read_csv('your_medical_data.csv')
    # Expected columns: 'DHS Code', 'Description', 'explanation'

    # Train distilled model
    trainer, student_model, tokenizer = train_distillation_model(
        df=df,
        teacher_model_id="microsoft/Phi-4-mini-instruct",
        teacher_adapter_path="./phi4-medical-lora",
        output_dir="./distilled-medical-explainer"
    )

    # Test the distilled model
    prompt = "Explain the medical code A00.0: Cholera due to Vibrio cholerae\nExplanation:"
    inputs = tokenizer(prompt, return_tensors="pt")

    with torch.no_grad():
        outputs = student_model.generate(
            inputs.input_ids,
            max_new_tokens=80,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    print(f"Student model explanation: {response}")


In [18]:
# Debug vocabulary sizes
print("=== VOCABULARY SIZE DEBUG ===")

# Check teacher tokenizer
print(f"Teacher tokenizer vocab size: {len(teacher_tokenizer)}")
print(f"Teacher tokenizer vocab_size attr: {teacher_tokenizer.vocab_size}")

# Check teacher model
print(f"Teacher model vocab size: {teacher_model.config.vocab_size}")
print(f"Teacher model embedding size: {teacher_model.get_input_embeddings().num_embeddings}")

# Check student model
print(f"Student model vocab size: {student_model.config.vocab_size}")
print(f"Student model embedding size: {student_model.get_input_embeddings().num_embeddings}")

# Check if they match
print(f"Teacher-Student vocab match: {teacher_model.config.vocab_size == student_model.config.vocab_size}")

=== VOCABULARY SIZE DEBUG ===


TypeError: object of type 'NoneType' has no len()

In [19]:
trainer, student_model, tokenizer = train_distillation_model(
        df=df,
        teacher_model_id="microsoft/Phi-4-mini-instruct",
        teacher_adapter_path= path + "finetuned/checkpoint-200/",
        output_dir= path + "distilled_model/"
    )

🚀 Starting Medical Explanation Model Distillation
1. Creating student model...
Student model configuration:
  Total Parameters: 190.55M
  Embedding dim: 1024
  Layers: 11
  Attention heads: 16
  Context length: 512
2. Loading teacher model...
   Loading teacher model...


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

3. Setting up tokenizers...


The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


...

/tmp/ipython-input-9-2545034996.py:16: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `DistillationTrainer.__init__`. Use `processing_class` instead.
  super().__init__(**kwargs)


✅ Distillation trainer initialized:
   Alpha (KL weight): 0.7
   Temperature: 1.0
7. Starting distillation training...
   Dataset size: 1300
   Estimated training time: ~2-4 hours


wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
wandb: Currently logged in as: chiragml (chiragml-civicai-lab) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...
`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.


OutOfMemoryError: CUDA out of memory. Tried to allocate 400.00 MiB. GPU 0 has a total capacity of 14.74 GiB of which 280.12 MiB is free. Process 119727 has 14.46 GiB memory in use. Of the allocated memory 13.48 GiB is allocated by PyTorch, and 874.76 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [4]:
def load_finetuned_model(adapter_path, base_model):

  # Load the adapter config to get base model info
  peft_config = PeftConfig.from_pretrained(adapter_path)

  # Load base model
  base_model = AutoModelForCausalLM.from_pretrained(
      peft_config.base_model_name_or_path,
      torch_dtype=torch.bfloat16,
      device_map='auto',
      trust_remote_code=True
  )

  # Load and apply adapters
  model = PeftModel.from_pretrained(base_model, adapter_path)

  # Load tokenizer
  tokenizer = AutoTokenizer.from_pretrained(peft_config.base_model_name_or_path)
  if tokenizer.pad_token is None:
      tokenizer.pad_token = tokenizer.eos_token

  return model, tokenizer

In [ ]:
load_finetuned_model(path + 'finetuned/checkpoint-200',None)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:2397: UserWarning: for base_model.model.model.layers.10.self_attn.o_proj.lora_A.default.weight: copying from a non-meta parameter in the checkpoint to a meta parameter in the current model, which is a no-op. (Did you mean to pass `assign=True` to assign items in the state dictionary to their corresponding key in the module instead of copying them in place?)
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:2397: UserWarning: for base_model.model.model.layers.10.self_attn.o_proj.lora_B.default.weight: copying from a non-meta parameter in the checkpoint to a meta parameter in the current model, which is a no-op. (Did you mean to pass `assign=True` to assign items in the state dictionary to their corresponding key in the module instead of copying them in place?)
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:2397: UserWarning: for base_model.model.model

KeyError: 'base_model.model.model.model.embed_tokens'

### Saving the finetunedmodel to huggingface

In [5]:
!pip install huggingface_hub

In [6]:
!huggingface-cli login


    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    A token is already saved on your machine. Run `huggingface-cli whoami` to get more information or `huggingface-cli logout` if you want to log out.
    Setting a new token will erase the existing one.
    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 
Add token as git credential? (Y/n) n
Token is valid (permission: write

In [7]:
def load_and_merge_finetuned_model(adapter_path, base_model):
    # Load the adapter config to get base model info
    peft_config = PeftConfig.from_pretrained(adapter_path)

    # Load base model
    base_model = AutoModelForCausalLM.from_pretrained(
        peft_config.base_model_name_or_path,
        torch_dtype=torch.bfloat16,
        device_map='auto',
        trust_remote_code=True
    )

    # Load and apply adapters
    model = PeftModel.from_pretrained(base_model, adapter_path)

    # **KEY STEP: Merge adapters with base model**
    model = model.merge_and_unload()

    # Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(peft_config.base_model_name_or_path)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    return model, tokenizer

# Load and merge the model
model, tokenizer = load_and_merge_finetuned_model(path + 'finetuned/checkpoint-200', None)



/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [8]:
## model, tokenizer = load_finetuned_model(path + 'finetuned/checkpoint-200',None)
model

Phi3ForCausalLM(
  (model): Phi3Model(
    (embed_tokens): Embedding(200064, 3072, padding_idx=199999)
    (layers): ModuleList(
      (0-31): 32 x Phi3DecoderLayer(
        (self_attn): Phi3Attention(
          (o_proj): Linear(in_features=3072, out_features=3072, bias=False)
          (qkv_proj): Linear(in_features=3072, out_features=5120, bias=False)
        )
        (mlp): Phi3MLP(
          (gate_up_proj): Linear(in_features=3072, out_features=16384, bias=False)
          (down_proj): Linear(in_features=8192, out_features=3072, bias=False)
          (activation_fn): SiLU()
        )
        (input_layernorm): Phi3RMSNorm((3072,), eps=1e-05)
        (post_attention_layernorm): Phi3RMSNorm((3072,), eps=1e-05)
        (resid_attn_dropout): Dropout(p=0.0, inplace=False)
        (resid_mlp_dropout): Dropout(p=0.0, inplace=False)
      )
    )
    (norm): Phi3RMSNorm((3072,), eps=1e-05)
    (rotary_emb): Phi3RotaryEmbedding()
  )
  (lm_head): Linear(in_features=3072, out_features=20006

In [9]:
model.push_to_hub("ChiragMl/phi4-medical-explainer")

model-00002-of-00002.safetensors:   0%|          | 0.00/2.77G [00:00<?, ?B/s]

Upload 2 LFS files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.90G [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/ChiragMl/phi4-medical-explainer/commit/eb6324d2793f07053f4d996ea362dcc467765899', commit_message='Upload Phi3ForCausalLM', commit_description='', oid='eb6324d2793f07053f4d996ea362dcc467765899', pr_url=None, repo_url=RepoUrl('https://huggingface.co/ChiragMl/phi4-medical-explainer', endpoint='https://huggingface.co', repo_type='model', repo_id='ChiragMl/phi4-medical-explainer'), pr_revision=None, pr_num=None)

In [10]:
tokenizer.push_to_hub("ChiragMl/phi4-medical-explainer")

tokenizer.json:   0%|          | 0.00/15.5M [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/ChiragMl/phi4-medical-explainer/commit/11bce5de022847a65a8b26b577bfa34e738d7885', commit_message='Upload tokenizer', commit_description='', oid='11bce5de022847a65a8b26b577bfa34e738d7885', pr_url=None, repo_url=RepoUrl('https://huggingface.co/ChiragMl/phi4-medical-explainer', endpoint='https://huggingface.co', repo_type='model', repo_id='ChiragMl/phi4-medical-explainer'), pr_revision=None, pr_num=None)

### Testing the upload

In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM

In [ ]:
test_model = AutoModelForCausalLM.from_pretrained("ChiragMl/phi4-medical-explainer")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [ ]:
test_tokenizer = AutoTokenizer.from_pretrained("ChiragMl/phi4-medical-explainer")

In [13]:
model.to('cuda')
model.device

device(type='cuda', index=0)

In [14]:
tokenizer.device = model.device

In [17]:
import torch

prompt = "Explain the medical code A00.0: Cholera due to Vibrio cholerae\nExplanation:"
inputs = tokenizer(prompt, return_tensors="pt")

# Get the device of the model
device = next(model.parameters()).device

# Move inputs to the same device as the model
inputs = {k: v.to(device) for k, v in inputs.items()}

# Generate using the full inputs dictionary
outputs = model.generate(
    **inputs,  # Use **inputs instead of just inputs.input_ids
    max_new_tokens=80,
    temperature=0.7,
    do_sample=True,
    pad_token_id=tokenizer.eos_token_id
)

response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
print(f"model explanation: {response}")

model explanation:  infection
